<a href="https://colab.research.google.com/github/Pes2ug23am092/AIS_transit_procurement_data_analysis/blob/main/Encephalon_ADA_eastcoast_NYNJ_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ---------------------------------------------------------------------------------------------
# CELL 1: SETUP, IMPORTS, AND DRIVE MOUNT
# ---------------------------------------------------------------------------------------------

# Install necessary libraries (Run this if you get 'module not found' errors)
# These installations may take a minute.
!pip install geopandas fiona shapely

# Essential imports
from google.colab import drive
import geopandas as gpd
import pandas as pd
import time
import os
import fiona
from shapely.geometry import shape
import gc
import psutil

# Mount Google Drive
drive.mount('/content/drive')

print("Setup and Drive Mount Complete.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup and Drive Mount Complete.


In [ ]:
# ---------------------------------------------------------------------------------------------
# CELL 2: MEMORY PREPARATION
# ---------------------------------------------------------------------------------------------

# Force garbage collection to free up any lingering memory
gc.collect()

# Report current memory usage
mem_usage = psutil.virtual_memory()
print(f"Current Memory Usage:")
print(f"  Available: {mem_usage.available / (1024**3):.2f} GB")
print(f"  Percent Used: {mem_usage.percent}%")

# Give a brief pause to allow resources to stabilize
time.sleep(5)

print("\nMemory preparation complete. You may now run the data extraction cells.")

Current Memory Usage:
  Available: 11.32 GB
  Percent Used: 10.7%

Memory preparation complete. You may now run the data extraction cells.


In [ ]:
# ---------------------------------------------------------------------------------------------
# CELL 3: 2021 EAST COAST DATA LOAD (NY/NJ BBOX)
# ---------------------------------------------------------------------------------------------

# --- Configuration ---
# File Path (based on your last provided code)
PATH_2021 = '/content/drive/MyDrive/ADA Encephalon/AIS_related/extracted/AISVesselTracks2021.gpkg'
# Output Directory (will hold 2021 data)
OUTPUT_PARQUET_DIR_2021 = '/content/drive/MyDrive/ADA Encephalon/AIS_related/war_processedoutput/2021'

# *** Stability Adjustment ***: Smaller chunk size to avoid memory crash
CHUNK_SIZE = 50000

# Columns needed: 'MMSI', 'TrackStartTime', 'Draft' (to get Draft_Meters), 'VesselType', 'geometry'
COLUMNS_TO_LOAD = ['MMSI', 'TrackStartTime', 'Draft', 'VesselType', 'geometry']

# *** Stability Adjustment ***: Tightly focused BBOX for faster test (New York Harbor)
BBOX_EAST_COAST = (-74.25, 40.5, -73.75, 41.0) # Approx. 0.5 degree area

# Ensure output directory exists
os.makedirs(OUTPUT_PARQUET_DIR_2021, exist_ok=True)
print(f"Output directory created: {OUTPUT_PARQUET_DIR_2021}")

# --- Chunked Reading, Filtering, and Saving ---
print(f"Starting chunked processing for 2021 East Coast data...")
start_total = time.time()
chunk_index = 0
total_features_processed = 0
output_crs = None
current_data = []

try:
    with fiona.open(PATH_2021, 'r', layer=0) as source:

        output_crs = source.crs

        # Iterate over features filtered by the BBOX
        for feature in source.filter(bbox=BBOX_EAST_COAST):

            # Extract properties (excluding geometry from properties)
            properties = {k: feature['properties'][k] for k in COLUMNS_TO_LOAD if k != 'geometry'}

            current_data.append({
                'geometry': shape(feature['geometry']),
                **properties
            })

            total_features_processed += 1

            # Process and save chunk
            if len(current_data) >= CHUNK_SIZE:
                chunk_index += 1
                start_chunk = time.time()

                gdf_chunk = gpd.GeoDataFrame(current_data, geometry='geometry', crs=output_crs)
                output_file = os.path.join(OUTPUT_PARQUET_DIR_2021, f'2021_east_coast_chunk_{chunk_index:03d}.parquet')
                gdf_chunk.to_parquet(output_file, index=False)

                print(f"   Saved Chunk {chunk_index} ({len(gdf_chunk):,} features) in {time.time() - start_chunk:.2f}s")

                current_data = []

    # Process the last, partial chunk
    if current_data:
        chunk_index += 1
        start_chunk = time.time()
        gdf_chunk = gpd.GeoDataFrame(current_data, geometry='geometry', crs=output_crs)
        output_file = os.path.join(OUTPUT_PARQUET_DIR_2021, f'2021_east_coast_chunk_{chunk_index:03d}.parquet')
        gdf_chunk.to_parquet(output_file, index=False)
        print(f"   Saved Final Chunk {chunk_index} ({len(gdf_chunk):,} features) in {time.time() - start_chunk:.2f}s")

    total_time = time.time() - start_total
    print(f"\n--- 2021 Processing Complete ---")
    print(f"Total features filtered and processed: {total_features_processed:,}")
    print(f"All relevant 2021 East Coast data is now saved to {OUTPUT_PARQUET_DIR_2021}")

except Exception as e:
    print(f"An error occurred during 2021 processing: {e}")

Output directory created: /content/drive/MyDrive/ADA Encephalon/AIS_related/war_processedoutput/2021
Starting chunked processing for 2021 East Coast data...


   Saved Chunk 1 (50,000 features) in 5.85s
   Saved Final Chunk 2 (44,708 features) in 2.43s

--- 2021 Processing Complete ---
Total features filtered and processed: 94,708
All relevant 2021 East Coast data is now saved to /content/drive/MyDrive/ADA Encephalon/AIS_related/war_processedoutput/2021


In [ ]:
# ---------------------------------------------------------------------------------------------
# CELL 4: 2021 DATA VIEW AND DRAFT_METERS CALCULATION
# ---------------------------------------------------------------------------------------------

PARQUET_DIR = '/content/drive/MyDrive/ADA Encephalon/AIS_related/war_processedoutput/2021'
# Assuming the first chunk is named '2021_east_coast_chunk_001.parquet'
parquet_file = os.path.join(PARQUET_DIR, '2021_east_coast_chunk_001.parquet')

if os.path.exists(parquet_file):
    print(f"Loading sample from: {parquet_file}")

    # Use geopandas to read the data
    gdf = gpd.read_parquet(parquet_file)

    # Convert Draft (in decimeters) to Draft_Meters
    if 'Draft' in gdf.columns:
        # 1. Clean up invalid/zero draft values
        gdf.loc[gdf['Draft'] <= 0.0, 'Draft'] = pd.NA
        # 2. Convert to meters (Draft is typically recorded in decimeters)
        gdf['Draft_Meters'] = gdf['Draft'] / 10.0
        # 3. Drop the original Draft column
        gdf.drop(columns=['Draft'], inplace=True)

    # Select the requested final columns
    COLUMNS_FINAL = ['MMSI', 'TrackStartTime', 'VesselType', 'Draft_Meters', 'geometry']

    # Filter and display a sample
    sample_data = gdf[COLUMNS_FINAL].head(10)

    print("\n--- Sample Data from 2021 East Coast Parquet (First 10 Rows) ---")

    # Display data without the complex geometry column for clarity
    print(sample_data.drop(columns=['geometry']).to_string())

else:
    print(f"Error: Parquet file not found at {parquet_file}. Run the previous cell first.")

Loading sample from: /content/drive/MyDrive/ADA Encephalon/AIS_related/war_processedoutput/2021/2021_east_coast_chunk_001.parquet

--- Sample Data from 2021 East Coast Parquet (First 10 Rows) ---
        MMSI             TrackStartTime  VesselType  Draft_Meters
0  636018028  2021-09-11T15:19:03+00:00        70.0          1.50
1  354315000  2021-08-19T12:44:41+00:00        70.0          1.45
2  249888000  2021-08-05T14:44:05+00:00        70.0          1.55
3  477168400  2021-08-19T12:46:04+00:00        71.0          1.18
4  477203100  2021-08-20T20:18:52+00:00        70.0          1.55
5  367596940  2021-08-05T15:51:13+00:00        31.0           NaN
6  367686010  2021-11-20T14:59:01+00:00        31.0          0.42
7  636016978  2021-08-05T13:32:56+00:00        70.0          1.17
8  211744000  2021-07-26T22:14:44+00:00        70.0          1.45
9  229626000  2021-07-09T08:16:01+00:00        71.0          1.21


In [ ]:
# ---------------------------------------------------------------------------------------------
# VERIFICATION CELL: Check for Geometry column
# ---------------------------------------------------------------------------------------------

import geopandas as gpd
import os

PARQUET_DIR = '/content/drive/MyDrive/ADA Encephalon/AIS_related/war_processedoutput/2021'
parquet_file = os.path.join(PARQUET_DIR, '2021_east_coast_chunk_001.parquet')

if os.path.exists(parquet_file):
    print(f"Loading sample from: {parquet_file}")

    # Use geopandas to read the data
    gdf = gpd.read_parquet(parquet_file)

    # Check the DataFrame structure
    print("\n--- DataFrame Structure ---")
    gdf.info()

    # You should see 'geometry' listed as a column with Dtype 'geometry'

else:
    print(f"Error: Parquet file not found at {parquet_file}.")


Loading sample from: /content/drive/MyDrive/ADA Encephalon/AIS_related/war_processedoutput/2021/2021_east_coast_chunk_001.parquet

--- DataFrame Structure ---
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   geometry        50000 non-null  geometry
 1   MMSI            50000 non-null  int64   
 2   TrackStartTime  50000 non-null  object  
 3   Draft           18721 non-null  float64 
 4   VesselType      47181 non-null  float64 
dtypes: float64(2), geometry(1), int64(1), object(1)
memory usage: 1.9+ MB


In [ ]:
# ---------------------------------------------------------------------------------------------
# CELL 6: 2022 EAST COAST DATA LOAD (War Period)
# ---------------------------------------------------------------------------------------------

# --- Configuration ---
PATH_2022 = '/content/drive/MyDrive/ADA Encephalon/AIS_related/extracted/AISVesselTracks2022.gpkg'
OUTPUT_PARQUET_DIR_2022 = '/content/drive/MyDrive/ADA Encephalon/AIS_related/war_processedoutput/2022'
CHUNK_SIZE = 50000 # Stable chunk size
COLUMNS_TO_LOAD = ['MMSI', 'TrackStartTime', 'Draft', 'VesselType', 'geometry']
BBOX_EAST_COAST = (-74.25, 40.5, -73.75, 41.0) # Focused NY/NJ Port area

# Ensure output directory exists
os.makedirs(OUTPUT_PARQUET_DIR_2022, exist_ok=True)
print(f"Output directory created: {OUTPUT_PARQUET_DIR_2022}")

# --- Chunked Reading, Filtering, and Saving ---
print(f"Starting chunked processing for 2022 East Coast data...")
start_total = time.time()
chunk_index = 0
total_features_processed = 0
output_crs = None
current_data = []

try:
    with fiona.open(PATH_2022, 'r', layer=0) as source:
        output_crs = source.crs
        for feature in source.filter(bbox=BBOX_EAST_COAST):
            properties = {k: feature['properties'][k] for k in COLUMNS_TO_LOAD if k != 'geometry'}
            current_data.append({'geometry': shape(feature['geometry']), **properties})
            total_features_processed += 1
            if len(current_data) >= CHUNK_SIZE:
                chunk_index += 1
                start_chunk = time.time()
                gdf_chunk = gpd.GeoDataFrame(current_data, geometry='geometry', crs=output_crs)
                output_file = os.path.join(OUTPUT_PARQUET_DIR_2022, f'2022_east_coast_chunk_{chunk_index:03d}.parquet')
                gdf_chunk.to_parquet(output_file, index=False)
                print(f"   Saved Chunk {chunk_index} ({len(gdf_chunk):,} features) in {time.time() - start_chunk:.2f}s")
                current_data = []

    if current_data:
        chunk_index += 1
        start_chunk = time.time()
        gdf_chunk = gpd.GeoDataFrame(current_data, geometry='geometry', crs=output_crs)
        output_file = os.path.join(OUTPUT_PARQUET_DIR_2022, f'2022_east_coast_chunk_{chunk_index:03d}.parquet')
        gdf_chunk.to_parquet(output_file, index=False)
        print(f"   Saved Final Chunk {chunk_index} ({len(gdf_chunk):,} features) in {time.time() - start_chunk:.2f}s")

    total_time = time.time() - start_total
    print(f"\n--- 2022 Processing Complete ---")
    print(f"Total features filtered and processed: {total_features_processed:,}")

except Exception as e:
    print(f"An error occurred during 2022 processing: {e}")

Output directory created: /content/drive/MyDrive/ADA Encephalon/AIS_related/war_processedoutput/2022
Starting chunked processing for 2022 East Coast data...


   Saved Chunk 1 (50,000 features) in 4.34s
   Saved Chunk 2 (50,000 features) in 4.33s
   Saved Final Chunk 3 (6,119 features) in 0.13s

--- 2022 Processing Complete ---
Total features filtered and processed: 106,119


In [ ]:
# ---------------------------------------------------------------------------------------------
# CELL 4: 2022 DATA VIEW AND DRAFT_METERS CALCULATION
# ---------------------------------------------------------------------------------------------

PARQUET_DIR = '/content/drive/MyDrive/ADA Encephalon/AIS_related/war_processedoutput/2022'
# Assuming the first chunk is named '2021_east_coast_chunk_001.parquet'
parquet_file = os.path.join(PARQUET_DIR, '2022_east_coast_chunk_001.parquet')

if os.path.exists(parquet_file):
    print(f"Loading sample from: {parquet_file}")

    # Use geopandas to read the data
    gdf = gpd.read_parquet(parquet_file)

    # Convert Draft (in decimeters) to Draft_Meters
    if 'Draft' in gdf.columns:
        # 1. Clean up invalid/zero draft values
        gdf.loc[gdf['Draft'] <= 0.0, 'Draft'] = pd.NA
        # 2. Convert to meters (Draft is typically recorded in decimeters)
        gdf['Draft_Meters'] = gdf['Draft'] / 10.0
        # 3. Drop the original Draft column
        gdf.drop(columns=['Draft'], inplace=True)

    # Select the requested final columns
    COLUMNS_FINAL = ['MMSI', 'TrackStartTime', 'VesselType', 'Draft_Meters', 'geometry']

    # Filter and display a sample
    sample_data = gdf[COLUMNS_FINAL].head(10)

    print("\n--- Sample Data from 2022 East Coast Parquet (First 10 Rows) ---")

    # Display data without the complex geometry column for clarity
    print(sample_data.drop(columns=['geometry']).to_string())

else:
    print(f"Error: Parquet file not found at {parquet_file}. Run the previous cell first.")

Loading sample from: /content/drive/MyDrive/ADA Encephalon/AIS_related/war_processedoutput/2022/2022_east_coast_chunk_001.parquet

--- Sample Data from 2022 East Coast Parquet (First 10 Rows) ---
        MMSI             TrackStartTime  VesselType  Draft_Meters
0  367460820  2022-05-29T15:00:40+00:00        37.0           NaN
1  338164158  2022-07-15T00:41:16+00:00        37.0           NaN
2  368224850  2022-04-13T11:35:58+00:00        65.0          0.30
3  338429597  2022-07-04T13:58:23+00:00        37.0           NaN
4  367036190  2022-05-31T06:01:15+00:00        40.0          0.21
5  338336726  2022-08-18T17:27:35+00:00        37.0           NaN
6  368224850  2022-08-25T23:09:53+00:00        65.0          0.30
7  367673370  2022-05-02T15:07:02+00:00        60.0          0.14
8  367673370  2022-04-26T02:20:26+00:00        60.0          0.14
9  367347130  2022-04-05T00:04:50+00:00        60.0          0.20


In [ ]:
# ---------------------------------------------------------------------------------------------
# CELL 7: 2023 EAST COAST DATA LOAD (Ongoing War Period)
# ---------------------------------------------------------------------------------------------

# --- Configuration ---
PATH_2023 = '/content/drive/MyDrive/ADA Encephalon/AIS_related/extracted/AISVesselTracks2023.gpkg'
OUTPUT_PARQUET_DIR_2023 = '/content/drive/MyDrive/ADA Encephalon/AIS_related/war_processedoutput/2023'
CHUNK_SIZE = 50000 # Stable chunk size
COLUMNS_TO_LOAD = ['MMSI', 'TrackStartTime', 'Draft', 'VesselType', 'geometry']
BBOX_EAST_COAST = (-74.25, 40.5, -73.75, 41.0) # Focused NY/NJ Port area

# Ensure output directory exists
os.makedirs(OUTPUT_PARQUET_DIR_2023, exist_ok=True)
print(f"Output directory created: {OUTPUT_PARQUET_DIR_2023}")

# --- Chunked Reading, Filtering, and Saving ---
print(f"Starting chunked processing for 2023 East Coast data...")
start_total = time.time()
chunk_index = 0
total_features_processed = 0
output_crs = None
current_data = []

try:
    with fiona.open(PATH_2023, 'r', layer=0) as source:
        output_crs = source.crs
        for feature in source.filter(bbox=BBOX_EAST_COAST):
            properties = {k: feature['properties'][k] for k in COLUMNS_TO_LOAD if k != 'geometry'}
            current_data.append({'geometry': shape(feature['geometry']), **properties})
            total_features_processed += 1
            if len(current_data) >= CHUNK_SIZE:
                chunk_index += 1
                start_chunk = time.time()
                gdf_chunk = gpd.GeoDataFrame(current_data, geometry='geometry', crs=output_crs)
                output_file = os.path.join(OUTPUT_PARQUET_DIR_2023, f'2023_east_coast_chunk_{chunk_index:03d}.parquet')
                gdf_chunk.to_parquet(output_file, index=False)
                print(f"   Saved Chunk {chunk_index} ({len(gdf_chunk):,} features) in {time.time() - start_chunk:.2f}s")
                current_data = []

    if current_data:
        chunk_index += 1
        start_chunk = time.time()
        gdf_chunk = gpd.GeoDataFrame(current_data, geometry='geometry', crs=output_crs)
        output_file = os.path.join(OUTPUT_PARQUET_DIR_2023, f'2023_east_coast_chunk_{chunk_index:03d}.parquet')
        gdf_chunk.to_parquet(output_file, index=False)
        print(f"   Saved Final Chunk {chunk_index} ({len(gdf_chunk):,} features) in {time.time() - start_chunk:.2f}s")

    total_time = time.time() - start_total
    print(f"\n--- 2023 Processing Complete ---")
    print(f"Total features filtered and processed: {total_features_processed:,}")

except Exception as e:
    print(f"An error occurred during 2023 processing: {e}")

Output directory created: /content/drive/MyDrive/ADA Encephalon/AIS_related/war_processedoutput/2023
Starting chunked processing for 2023 East Coast data...


   Saved Chunk 1 (50,000 features) in 2.46s
   Saved Chunk 2 (50,000 features) in 7.77s
   Saved Final Chunk 3 (7,377 features) in 0.93s

--- 2023 Processing Complete ---
Total features filtered and processed: 107,377


In [ ]:
# ---------------------------------------------------------------------------------------------
# CELL 4: 2023 DATA VIEW AND DRAFT_METERS CALCULATION
# ---------------------------------------------------------------------------------------------

PARQUET_DIR = '/content/drive/MyDrive/ADA Encephalon/AIS_related/war_processedoutput/2023'
# Assuming the first chunk is named '2021_east_coast_chunk_001.parquet'
parquet_file = os.path.join(PARQUET_DIR, '2023_east_coast_chunk_001.parquet')

if os.path.exists(parquet_file):
    print(f"Loading sample from: {parquet_file}")

    # Use geopandas to read the data
    gdf = gpd.read_parquet(parquet_file)

    # Convert Draft (in decimeters) to Draft_Meters
    if 'Draft' in gdf.columns:
        # 1. Clean up invalid/zero draft values
        gdf.loc[gdf['Draft'] <= 0.0, 'Draft'] = pd.NA
        # 2. Convert to meters (Draft is typically recorded in decimeters)
        gdf['Draft_Meters'] = gdf['Draft'] / 10.0
        # 3. Drop the original Draft column
        gdf.drop(columns=['Draft'], inplace=True)

    # Select the requested final columns
    COLUMNS_FINAL = ['MMSI', 'TrackStartTime', 'VesselType', 'Draft_Meters', 'geometry']

    # Filter and display a sample
    sample_data = gdf[COLUMNS_FINAL].head(10)

    print("\n--- Sample Data from 2023 East Coast Parquet (First 10 Rows) ---")

    # Display data without the complex geometry column for clarity
    print(sample_data.drop(columns=['geometry']).to_string())

else:
    print(f"Error: Parquet file not found at {parquet_file}. Run the previous cell first.")

Loading sample from: /content/drive/MyDrive/ADA Encephalon/AIS_related/war_processedoutput/2023/2023_east_coast_chunk_001.parquet

--- Sample Data from 2023 East Coast Parquet (First 10 Rows) ---
        MMSI             TrackStartTime  VesselType  Draft_Meters
0  367010390  2023-12-12T20:05:54+00:00        31.0           NaN
1  367314640  2023-12-01T00:00:02+00:00        31.0          0.42
2  369493610  2023-12-11T19:06:10+00:00        90.0           NaN
3  367165430  2023-12-10T06:08:44+00:00        31.0          0.39
4  367509760  2023-12-07T12:57:58+00:00        90.0           NaN
5  367770270  2023-12-12T20:05:51+00:00        31.0           NaN
6  366999414  2023-12-15T14:02:07+00:00        90.0           NaN
7  369493610  2023-12-29T19:32:45+00:00        90.0           NaN
8  368396216  2023-12-22T13:32:46+00:00        40.0           NaN
9  338450388  2023-12-18T13:42:55+00:00        31.0          0.10


In [ ]:
# ---------------------------------------------------------------------------------------------
# CELL 7: 2024 EAST COAST DATA LOAD (Ongoing War Period)
# ---------------------------------------------------------------------------------------------

# --- Configuration ---
PATH_2024 = '/content/drive/MyDrive/ADA Encephalon/AIS_related/extracted/AISVesselTracks2024.gpkg'
OUTPUT_PARQUET_DIR_2024 = '/content/drive/MyDrive/ADA Encephalon/AIS_related/war_processedoutput/2024'
CHUNK_SIZE = 50000 # Stable chunk size
COLUMNS_TO_LOAD = ['MMSI', 'TrackStartTime', 'Draft', 'VesselType', 'geometry']
BBOX_EAST_COAST = (-74.25, 40.5, -73.75, 41.0) # Focused NY/NJ Port area

# Ensure output directory exists
os.makedirs(OUTPUT_PARQUET_DIR_2024, exist_ok=True)
print(f"Output directory created: {OUTPUT_PARQUET_DIR_2024}")

# --- Chunked Reading, Filtering, and Saving ---
print(f"Starting chunked processing for 2024 East Coast data...")
start_total = time.time()
chunk_index = 0
total_features_processed = 0
output_crs = None
current_data = []

try:
    with fiona.open(PATH_2024, 'r', layer=0) as source:
        output_crs = source.crs
        for feature in source.filter(bbox=BBOX_EAST_COAST):
            properties = {k: feature['properties'][k] for k in COLUMNS_TO_LOAD if k != 'geometry'}
            current_data.append({'geometry': shape(feature['geometry']), **properties})
            total_features_processed += 1
            if len(current_data) >= CHUNK_SIZE:
                chunk_index += 1
                start_chunk = time.time()
                gdf_chunk = gpd.GeoDataFrame(current_data, geometry='geometry', crs=output_crs)
                output_file = os.path.join(OUTPUT_PARQUET_DIR_2024, f'2024_east_coast_chunk_{chunk_index:03d}.parquet')
                gdf_chunk.to_parquet(output_file, index=False)
                print(f"   Saved Chunk {chunk_index} ({len(gdf_chunk):,} features) in {time.time() - start_chunk:.2f}s")
                current_data = []

    if current_data:
        chunk_index += 1
        start_chunk = time.time()
        gdf_chunk = gpd.GeoDataFrame(current_data, geometry='geometry', crs=output_crs)
        output_file = os.path.join(OUTPUT_PARQUET_DIR_2024, f'2024_east_coast_chunk_{chunk_index:03d}.parquet')
        gdf_chunk.to_parquet(output_file, index=False)
        print(f"   Saved Final Chunk {chunk_index} ({len(gdf_chunk):,} features) in {time.time() - start_chunk:.2f}s")

    total_time = time.time() - start_total
    print(f"\n--- 2024 Processing Complete ---")
    print(f"Total features filtered and processed: {total_features_processed:,}")

except Exception as e:
    print(f"An error occurred during 2024 processing: {e}")

Output directory created: /content/drive/MyDrive/ADA Encephalon/AIS_related/war_processedoutput/2024
Starting chunked processing for 2024 East Coast data...


   Saved Chunk 1 (50,000 features) in 3.71s
   Saved Chunk 2 (50,000 features) in 1.91s
   Saved Final Chunk 3 (22,600 features) in 3.07s

--- 2024 Processing Complete ---
Total features filtered and processed: 122,600


In [ ]:
# ---------------------------------------------------------------------------------------------
# CELL 4: 2024 DATA VIEW AND DRAFT_METERS CALCULATION
# ---------------------------------------------------------------------------------------------

PARQUET_DIR = '/content/drive/MyDrive/ADA Encephalon/AIS_related/war_processedoutput/2024'
# Assuming the first chunk is named '2021_east_coast_chunk_001.parquet'
parquet_file = os.path.join(PARQUET_DIR, '2024_east_coast_chunk_001.parquet')

if os.path.exists(parquet_file):
    print(f"Loading sample from: {parquet_file}")

    # Use geopandas to read the data
    gdf = gpd.read_parquet(parquet_file)

    # Convert Draft (in decimeters) to Draft_Meters
    if 'Draft' in gdf.columns:
        # 1. Clean up invalid/zero draft values
        gdf.loc[gdf['Draft'] <= 0.0, 'Draft'] = pd.NA
        # 2. Convert to meters (Draft is typically recorded in decimeters)
        gdf['Draft_Meters'] = gdf['Draft'] / 10.0
        # 3. Drop the original Draft column
        gdf.drop(columns=['Draft'], inplace=True)

    # Select the requested final columns
    COLUMNS_FINAL = ['MMSI', 'TrackStartTime', 'VesselType', 'Draft_Meters', 'geometry']

    # Filter and display a sample
    sample_data = gdf[COLUMNS_FINAL].head(10)

    print("\n--- Sample Data from 2024 East Coast Parquet (First 10 Rows) ---")

    # Display data without the complex geometry column for clarity
    print(sample_data.drop(columns=['geometry']).to_string())

else:
    print(f"Error: Parquet file not found at {parquet_file}. Run the previous cell first.")

Loading sample from: /content/drive/MyDrive/ADA Encephalon/AIS_related/war_processedoutput/2024/2024_east_coast_chunk_001.parquet

--- Sample Data from 2024 East Coast Parquet (First 10 Rows) ---
        MMSI             TrackStartTime  VesselType  Draft_Meters
0  477462400  2024-03-10T09:34:43+00:00        70.0          1.06
1  368247000  2024-08-03T02:34:10+00:00        52.0          0.67
2  368247000  2024-06-18T22:00:03+00:00        52.0          0.67
3  367728850  2024-07-21T14:09:57+00:00        52.0          0.68
4  368051050  2024-02-12T15:31:58+00:00        30.0          0.53
5  367596940  2024-08-01T00:00:10+00:00        52.0          0.37
6  367006570  2024-05-23T06:09:18+00:00        52.0          0.50
7  636016436  2024-03-10T07:46:26+00:00        70.0          1.09
8  367185680  2024-05-20T04:24:17+00:00        52.0          0.42
9  366997290  2024-07-04T14:05:17+00:00        60.0           NaN


In [ ]:
# ---------------------------------------------------------------------------------------------
# CELL 5: 2020 EAST COAST DATA LOAD (REVISED FOR .GDB FORMAT)
# ---------------------------------------------------------------------------------------------

# --- Configuration ---
# *** ADJUSTMENT 1: Change file extension to .gdb ***
PATH_2020 = '/content/drive/MyDrive/ADA Encephalon/AIS_related/extracted/AISVesselTracks2020.gdb'
OUTPUT_PARQUET_DIR_2020 = '/content/drive/MyDrive/ADA Encephalon/AIS_related/war_processedoutput/2020'
CHUNK_SIZE = 50000 # Stable chunk size
COLUMNS_TO_LOAD = ['MMSI', 'TrackStartTime', 'Draft', 'VesselType', 'geometry']
BBOX_EAST_COAST = (-74.25, 40.5, -73.75, 41.0) # Focused NY/NJ Port area

# *** ADJUSTMENT 2: Specify the layer name for the .gdb file ***
GDB_LAYER_NAME = 'AISVesselTracks2020'


# Ensure output directory exists
os.makedirs(OUTPUT_PARQUET_DIR_2020, exist_ok=True)
print(f"Output directory created: {OUTPUT_PARQUET_DIR_2020}")

# --- Chunked Reading, Filtering, and Saving ---
print(f"Starting chunked processing for 2020 East Coast data from GDB...")
start_total = time.time()
chunk_index = 0
total_features_processed = 0
output_crs = None
current_data = []

try:
    # --- IMPORTANT CHANGE: Specify layer=GDB_LAYER_NAME ---
    with fiona.open(PATH_2020, 'r', layer=GDB_LAYER_NAME) as source:

        output_crs = source.crs

        for feature in source.filter(bbox=BBOX_EAST_COAST):
            properties = {k: feature['properties'][k] for k in COLUMNS_TO_LOAD if k != 'geometry'}
            current_data.append({'geometry': shape(feature['geometry']), **properties})
            total_features_processed += 1

            if len(current_data) >= CHUNK_SIZE:
                chunk_index += 1
                start_chunk = time.time()
                gdf_chunk = gpd.GeoDataFrame(current_data, geometry='geometry', crs=output_crs)
                output_file = os.path.join(OUTPUT_PARQUET_DIR_2020, f'2020_east_coast_chunk_{chunk_index:03d}.parquet')
                gdf_chunk.to_parquet(output_file, index=False)
                print(f"   Saved Chunk {chunk_index} ({len(gdf_chunk):,} features) in {time.time() - start_chunk:.2f}s")
                current_data = []

    if current_data:
        chunk_index += 1
        start_chunk = time.time()
        gdf_chunk = gpd.GeoDataFrame(current_data, geometry='geometry', crs=output_crs)
        output_file = os.path.join(OUTPUT_PARQUET_DIR_2020, f'2020_east_coast_chunk_{chunk_index:03d}.parquet')
        gdf_chunk.to_parquet(output_file, index=False)
        print(f"   Saved Final Chunk {chunk_index} ({len(gdf_chunk):,} features) in {time.time() - start_chunk:.2f}s")

    total_time = time.time() - start_total
    print(f"\n--- 2020 Processing Complete ---")
    print(f"Total features filtered and processed: {total_features_processed:,}")

except Exception as e:
    print(f"An error occurred during 2020 GDB processing: {e}")

Output directory created: /content/drive/MyDrive/ADA Encephalon/AIS_related/war_processedoutput/2020
Starting chunked processing for 2020 East Coast data from GDB...
   Saved Chunk 1 (50,000 features) in 9.38s
   Saved Chunk 2 (50,000 features) in 1.92s
   Saved Final Chunk 3 (13,359 features) in 1.36s

--- 2020 Processing Complete ---
Total features filtered and processed: 113,359


In [ ]:
# ---------------------------------------------------------------------------------------------
# CELL 4: 2020 DATA VIEW AND DRAFT_METERS CALCULATION
# ---------------------------------------------------------------------------------------------

PARQUET_DIR = '/content/drive/MyDrive/ADA Encephalon/AIS_related/war_processedoutput/2020'
# Assuming the first chunk is named '2021_east_coast_chunk_001.parquet'
parquet_file = os.path.join(PARQUET_DIR, '2020_east_coast_chunk_001.parquet')

if os.path.exists(parquet_file):
    print(f"Loading sample from: {parquet_file}")

    # Use geopandas to read the data
    gdf = gpd.read_parquet(parquet_file)

    # Convert Draft (in decimeters) to Draft_Meters
    if 'Draft' in gdf.columns:
        # 1. Clean up invalid/zero draft values
        gdf.loc[gdf['Draft'] <= 0.0, 'Draft'] = pd.NA
        # 2. Convert to meters (Draft is typically recorded in decimeters)
        gdf['Draft_Meters'] = gdf['Draft'] / 10.0
        # 3. Drop the original Draft column
        gdf.drop(columns=['Draft'], inplace=True)

    # Select the requested final columns
    COLUMNS_FINAL = ['MMSI', 'TrackStartTime', 'VesselType', 'Draft_Meters', 'geometry']

    # Filter and display a sample
    sample_data = gdf[COLUMNS_FINAL].head(10)

    print("\n--- Sample Data from 2024 East Coast Parquet (First 10 Rows) ---")

    # Display data without the complex geometry column for clarity
    print(sample_data.drop(columns=['geometry']).to_string())

else:
    print(f"Error: Parquet file not found at {parquet_file}. Run the previous cell first.")

Loading sample from: /content/drive/MyDrive/ADA Encephalon/AIS_related/war_processedoutput/2020/2020_east_coast_chunk_001.parquet

--- Sample Data from 2024 East Coast Parquet (First 10 Rows) ---
        MMSI             TrackStartTime  VesselType  Draft_Meters
0  211367460  2020-01-14T10:33:22+00:00        70.0          1.45
1  211367460  2020-01-14T13:45:28+00:00        70.0          1.45
2  215137000  2020-01-17T12:55:20+00:00         NaN           NaN
3  215137000  2020-01-20T10:30:11+00:00         NaN           NaN
4  215137000  2020-01-20T10:41:42+00:00         NaN           NaN
5  215251000  2020-01-25T06:38:04+00:00        70.0           NaN
6  215251000  2020-01-27T13:45:19+00:00        70.0           NaN
7  218063000  2020-01-24T09:57:16+00:00        70.0          1.46
8  218063000  2020-01-24T19:34:52+00:00        70.0          1.46
9  218843000  2020-01-27T21:53:57+00:00        70.0          1.50


In [ ]:
# ---------------------------------------------------------------------------------------------
# CELL 9: COMBINE, CLEAN, AND SAVE FINAL DATASET (2020-2024)
# ---------------------------------------------------------------------------------------------

# Base directory where all yearly processed folders are stored
BASE_PROCESSED_DIR = '/content/drive/MyDrive/ADA Encephalon/AIS_related/war_processedoutput'

# List of all years to combine (2020-2024)
YEARS_TO_COMBINE = ['2020', '2021', '2022', '2023', '2024']

# Final output file path
FINAL_OUTPUT_FILE = os.path.join(BASE_PROCESSED_DIR, 'combined_east_coast_NYNJ_2020_2024_clean.parquet')

# List to hold all loaded GeoDataFrames
all_gdfs = []
total_combined_rows = 0

print("Starting combination and cleaning of all yearly data...")
start_total = time.time()

for year in YEARS_TO_COMBINE:
    year_dir = os.path.join(BASE_PROCESSED_DIR, year)

    if os.path.isdir(year_dir):
        # Read all Parquet files in the yearly directory
        parquet_files = [os.path.join(year_dir, f) for f in os.listdir(year_dir) if f.endswith('.parquet')]

        if parquet_files:
            # Read and combine all chunks for the current year
            year_gdfs = [gpd.read_parquet(f) for f in parquet_files]

            # Concatenate the chunks for the year
            gdf_year = pd.concat(year_gdfs, ignore_index=True)

            # --- Apply Final Data Cleaning (Draft_Meters Conversion) ---
            if 'Draft' in gdf_year.columns:
                # 1. Clean invalid/zero draft values
                gdf_year.loc[gdf_year['Draft'] <= 0.0, 'Draft'] = pd.NA
                # 2. Convert to meters (Draft is in decimeters)
                gdf_year['Draft_Meters'] = gdf_year['Draft'] / 10.0
                # 3. Drop the original Draft column
                gdf_year.drop(columns=['Draft'], inplace=True)

            # Filter to final required columns (MMSI, TrackStartTime, VesselType, Draft_Meters, geometry)
            gdf_year = gdf_year[['MMSI', 'TrackStartTime', 'VesselType', 'Draft_Meters', 'geometry']]

            total_combined_rows += len(gdf_year)
            all_gdfs.append(gdf_year)
            print(f"   ✅ Loaded and Cleaned {len(gdf_year):,} features for year {year}.")

        else:
            print(f"   ⚠️ Warning: No Parquet files found in directory for year {year}: {year_dir}")
    else:
        print(f"   ❌ Error: Directory not found for year {year}: {year_dir}")

# Final concatenation of all years
if all_gdfs:
    final_combined_gdf = pd.concat(all_gdfs, ignore_index=True)

    # Save the final combined dataset as GeoParquet
    final_combined_gdf.to_parquet(FINAL_OUTPUT_FILE, index=False)

    total_time = time.time() - start_total
    print(f"\n--- Combination Complete ---")
    print(f"Total features in final dataset: {len(final_combined_gdf):,}")
    print(f"Total time taken: {total_time:.2f} seconds")
    print(f"Final cleaned dataset saved to: {FINAL_OUTPUT_FILE}")
    print("\nNext step: Analyze the data by grouping by month and year.")
else:
    print("\n--- Combination Failed ---")
    print("No data was combined. Check your yearly directories and file paths.")

Starting combination and cleaning of all yearly data...
   ✅ Loaded and Cleaned 113,359 features for year 2020.
   ✅ Loaded and Cleaned 50,000 features for year 2021.
   ✅ Loaded and Cleaned 106,119 features for year 2022.
   ✅ Loaded and Cleaned 107,377 features for year 2023.
   ✅ Loaded and Cleaned 122,600 features for year 2024.

--- Combination Complete ---
Total features in final dataset: 499,455
Total time taken: 301.49 seconds
Final cleaned dataset saved to: /content/drive/MyDrive/ADA Encephalon/AIS_related/war_processedoutput/combined_east_coast_NYNJ_2020_2024_clean.parquet

Next step: Analyze the data by grouping by month and year.
